In [0]:
from pyspark.sql import functions as F

CATALOGO = "workspace"
ESQUEMA = "ingenieria_datos_juanjo"
T_BRONZE = f"{CATALOGO}.{ESQUEMA}.ventas_bronze"
T_SILVER = f"{CATALOGO}.{ESQUEMA}.ventas_silver"
T_CAT    = f"{CATALOGO}.{ESQUEMA}.ventas_por_categoria"

# 1. Leer bronze
bronze = spark.read.table(T_BRONZE)
n_bronze = bronze.count()

# 2. Eliminar registros idénticos duplicados
sin_duplicados = bronze.dropDuplicates()
n_sin_dup = sin_duplicados.count()

# 3. Reglas de calidad: producto presente, cantidad > 0, precio > 0
validas = sin_duplicados.filter(
    F.col("producto").isNotNull()
    & (F.trim(F.col("producto")) != "")
    & (F.col("cantidad") > 0)
    & (F.col("precio_unitario") > 0)
)

# 4. Columna calculada
silver = validas.withColumn("total_venta", F.col("cantidad") * F.col("precio_unitario"))
n_silver = silver.count()

# 5. Guardar silver (idempotente)
(silver.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(T_SILVER))

# 6. Resumen por categoría (idempotente)
por_categoria = (spark.read.table(T_SILVER)
    .groupBy("categoria")
    .agg(F.sum("total_venta").alias("total_vendido"),
         F.count("*").alias("num_ventas"))
    .orderBy(F.desc("total_vendido")))
(por_categoria.write.format("delta").mode("overwrite")
              .option("overwriteSchema", "true").saveAsTable(T_CAT))

# Reporte de calidad
print(f"Registros en bronze:            {n_bronze}")
print(f"Tras quitar duplicados:         {n_sin_dup}  (duplicados eliminados: {n_bronze - n_sin_dup})")
print(f"Tras reglas de calidad (silver): {n_silver}  (descartados: {n_sin_dup - n_silver})")
display(spark.read.table(T_CAT))

Registros en bronze:            17
Tras quitar duplicados:         16  (duplicados eliminados: 1)
Tras reglas de calidad (silver): 14  (descartados: 2)


categoria,total_vendido,num_ventas
Electrónica,9290000.0,6
Ropa,1450000.0,4
Accesorios,1295000.0,4


In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM workspace.ingenieria_datos_juanjo.ventas_silver)                       AS filas_silver,
  (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM workspace.ingenieria_datos_juanjo.ventas_silver)) AS filas_silver_distintas,
  (SELECT COUNT(*) FROM workspace.ingenieria_datos_juanjo.ventas_por_categoria)                AS filas_categoria;

filas_silver,filas_silver_distintas,filas_categoria
14,14,3


In [0]:
%sql
 DESCRIBE HISTORY workspace.ingenieria_datos_juanjo.ventas_silver;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-11-11T22:44:48.000Z,75729600584658,juanjosejaramillo1414@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""316b3ad7-4a87-42e4-bc42-03376a6d4682"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-4739787d-c0d3-4519-ac1b-ffa40d06efbc"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-be7323e6-0dea-488a-9070-5fdd05f5b761""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(3954934393340866),5cfaf243-f905-4baa-92c6-7d820c34ed35,1111-222435-p6k96qr6-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 2774, numDeletionVectorsRemoved -> 0, numOutputRows -> 14, numOutputBytes -> 2774)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-11T22:40:09.000Z,75729600584658,juanjosejaramillo1414@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""316b3ad7-4a87-42e4-bc42-03376a6d4682"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-4739787d-c0d3-4519-ac1b-ffa40d06efbc"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-be7323e6-0dea-488a-9070-5fdd05f5b761""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(3954934393340866),fcf6200e-4df9-47d4-9ee6-a063bb4cd069,1111-222435-p6k96qr6-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 14, numOutputBytes -> 2774)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
